In [5]:

# ============================================================
# AFRIRAMP AI ASSISTANT — GEMMA 3 270M IT
# Kaggle full training and evaluation script
#
# Fixes:
# 1. FP16 gradient unscaling error: full FP32 training.
# 2. Blank generation: correct chat-template and token decoding.
# 3. FileNotFoundError: save files under /kaggle/working.
# 4. Includes synthetic training data and held-out evaluation.
# ============================================================

import os
import sys
import json
import csv
import gc
import zipfile
import random
import subprocess
from pathlib import Path
from datetime import datetime

# -----------------------------
# 1. Install required libraries
# -----------------------------

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.51,<5",
    "datasets>=3.2,<4",
    "accelerate>=1.2,<2",
    "huggingface_hub",
    "safetensors"
])

import torch
import pandas as pd

from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    set_seed
)

# -----------------------------
# 2. Configuration
# -----------------------------

SEED = 42
MODEL_NAME = "google/gemma-3-270m-it"

OUTPUT_DIR = Path("/kaggle/working/afriramp-gemma")
FINAL_MODEL_DIR = OUTPUT_DIR / "final_model"
RESULTS_DIR = OUTPUT_DIR / "results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 512
MAX_NEW_TOKENS = 120
EPOCHS = 4
LEARNING_RATE = 2e-5

set_seed(SEED)
random.seed(SEED)

print("=" * 65)
print("AFRIRAMP GEMMA TRAINING")
print("=" * 65)
print("Model:", MODEL_NAME)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
        "GB"
    )
else:
    print(
        "WARNING: No GPU detected. In Kaggle, enable a GPU "
        "accelerator for faster training."
    )

# -----------------------------
# 3. Hugging Face authentication
# -----------------------------

# Preferred: create a Kaggle Secret named HF_TOKEN.
# Alternatively, enter your token when prompted.
#
# Never paste your token into public notebooks or share it.

hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None

if not hf_token:
    from getpass import getpass
    hf_token = getpass("Enter your Hugging Face token: ")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication completed.")
else:
    raise RuntimeError(
        "A Hugging Face token is required to access Gemma."
    )

# -----------------------------
# 4. AfriRamp product context
# -----------------------------

AFRIRAMP_CONTEXT = """
You are AfriRamp Assistant, an AI assistant for the AfriRamp
venture.

AfriRamp is developing a digital financial service for African
freelancers and remote digital workers who receive payments from
international clients.

The problem AfriRamp aims to address includes expensive cross-border
payments, slow receipt of international earnings, fragmented payment
processes, and difficulty converting foreign earnings into usable
local currency.

AfriRamp is exploring technology-assisted payment workflows that
make receiving and managing international earnings more accessible.

The intended users include African freelancers, remote workers,
digital creators, and online professionals.

You should explain AfriRamp clearly, accurately, and in simple
language. Be helpful, professional, concise, and honest.

IMPORTANT LIMITATIONS:
- Do not claim AfriRamp has launched unless the user provides evidence.
- Do not invent active users, customers, revenue, transaction volumes,
  partnerships, licenses, regulatory approvals, or supported currencies.
- Do not invent fees, exchange rates, processing times, or guarantees.
- Do not claim a payment has been sent, received, or completed.
- Do not request passwords, private keys, seed phrases, or full
  authentication credentials.
- If asked for a specific fee, rate, supported country, or payment
  status that is not in the available information, say that it must
  be confirmed through official AfriRamp information.
- Explain that the product is under development where appropriate.
- Never present synthetic training examples as real customer records.
- AI can help with information and guidance, but must not be described
  as independently executing financial transactions.
"""

# -----------------------------
# 5. Synthetic training examples
# -----------------------------
# These are example conversations, not real customer records.

training_examples = [
    (
        "What is AfriRamp?",
        "AfriRamp is developing a digital financial service to help "
        "African freelancers and remote workers manage international "
        "earnings more conveniently."
    ),
    (
        "What problem does AfriRamp solve?",
        "AfriRamp aims to address expensive cross-border payments, "
        "slow payment receipt, and difficulties converting "
        "international earnings into usable local currency."
    ),
    (
        "Who is AfriRamp designed for?",
        "AfriRamp is designed primarily for African freelancers, "
        "remote workers, digital creators, and online professionals "
        "who receive payments from international clients."
    ),
    (
        "How can AfriRamp help freelancers?",
        "AfriRamp aims to make international earnings easier to "
        "receive and manage. Exact features and availability should "
        "be confirmed through official product information."
    ),
    (
        "Does AfriRamp use artificial intelligence?",
        "AfriRamp is exploring AI-assisted capabilities to improve "
        "user guidance and make financial workflows easier to "
        "understand. Specific features depend on product development."
    ),
    (
        "Can AfriRamp convert dollars to Ugandan shillings?",
        "Currency conversion availability has to be confirmed through "
        "official AfriRamp information. I cannot confirm that a "
        "specific currency pair is currently supported."
    ),
    (
        "What are AfriRamp's transaction fees?",
        "I do not have verified current AfriRamp fee information. "
        "Please check official product information before making "
        "a financial decision."
    ),
    (
        "How long do AfriRamp payments take?",
        "Processing times depend on the payment method, providers, "
        "and product availability. I cannot promise a processing "
        "time without verified information."
    ),
    (
        "Is AfriRamp available to everyone?",
        "Availability and supported countries need to be confirmed "
        "through official AfriRamp information."
    ),
    (
        "Is AfriRamp already launched?",
        "AfriRamp is being developed. I cannot confirm a public "
        "launch or general availability from the information "
        "currently available."
    ),
    (
        "Does AfriRamp have active customers?",
        "I do not have verified current customer numbers. I should "
        "not invent customer figures or claim traction without "
        "reliable evidence."
    ),
    (
        "How does AfriRamp make money?",
        "AfriRamp's final revenue model and pricing should be confirmed "
        "from its business plan. Potential financial-service revenue "
        "models must be evaluated transparently and responsibly."
    ),
    (
        "Is AfriRamp a bank?",
        "AfriRamp should not be described as a licensed bank unless "
        "that status is verified. Its exact legal and regulatory "
        "status must be confirmed through official documentation."
    ),
    (
        "Can I trust AfriRamp with my money?",
        "Before using any financial service, verify its legal status, "
        "security practices, fees, terms, and payment partners. "
        "Do not transfer money based only on an AI-generated answer."
    ),
    (
        "Can AfriRamp guarantee the best exchange rate?",
        "No such guarantee can be confirmed from the available "
        "information. Exchange rates vary and should be checked "
        "before a transaction."
    ),
    (
        "What makes AfriRamp useful in Africa?",
        "AfriRamp focuses on challenges faced by African remote "
        "workers and freelancers receiving international earnings. "
        "Its intended value is making these financial workflows "
        "more accessible and convenient."
    ),
    (
        "How do I contact AfriRamp support?",
        "Please use AfriRamp's verified official contact channels "
        "when they are available. I do not have a verified support "
        "email or telephone number to provide."
    ),
    (
        "Can AfriRamp send money right now?",
        "I cannot initiate or verify a real payment. Product "
        "availability and transaction capabilities must be confirmed "
        "through official AfriRamp channels."
    ),
    (
        "What should I do if a payment is delayed?",
        "Check the transaction status through the relevant official "
        "payment provider and retain your transaction reference. "
        "Never share your password, PIN, private key, or seed phrase."
    ),
    (
        "How does AfriRamp protect users?",
        "Users should look for clear security practices, secure "
        "authentication, transparent fees, privacy protections, "
        "and appropriate regulatory compliance. I cannot confirm "
        "specific implemented controls without verified documentation."
    ),
    (
        "Does AfriRamp support mobile money?",
        "I cannot confirm current mobile-money integrations. Please "
        "check official AfriRamp information for supported payment "
        "methods and countries."
    ),
    (
        "Can a student use AfriRamp?",
        "Eligibility depends on the service's eventual availability, "
        "supported countries, and account requirements. Those details "
        "should be verified through official product information."
    ),
    (
        "What is cross-border payment?",
        "A cross-border payment is a transfer of money between "
        "people or organizations in different countries."
    ),
    (
        "Why are international payments expensive?",
        "Costs can come from provider fees, intermediary banks, "
        "currency conversion spreads, and other processing charges. "
        "The actual cost depends on the payment route."
    ),
    (
        "What is a foreign exchange rate?",
        "A foreign exchange rate describes how much one currency "
        "is worth in another currency. Rates can change over time."
    ),
    (
        "What information should I check before sending money?",
        "Confirm the recipient, total fees, exchange rate, expected "
        "delivery time, refund terms, and provider legitimacy before "
        "authorizing a payment."
    ),
    (
        "Should I share my seed phrase with support?",
        "No. Never share your seed phrase or private keys with "
        "support, an AI assistant, or another person."
    ),
    (
        "Can AI make financial mistakes?",
        "Yes. AI can produce inaccurate information. Verify important "
        "financial details with official sources before acting."
    ),
    (
        "Can you confirm my AfriRamp balance?",
        "I cannot access or verify your account balance. Check the "
        "official account interface or contact verified support."
    ),
    (
        "What if I received a suspicious AfriRamp message?",
        "Do not click suspicious links or share sensitive information. "
        "Verify the message through an independently confirmed "
        "official contact channel."
    ),
    (
        "Can AfriRamp help me get paid by an international client?",
        "AfriRamp is intended to address international payment "
        "challenges for African remote workers. Confirm current "
        "availability and supported payment routes before relying "
        "on it for a payment."
    ),
    (
        "Does AfriRamp charge hidden fees?",
        "I cannot confirm AfriRamp's current pricing. Before using "
        "any financial service, request a clear breakdown of all "
        "fees and currency-conversion costs."
    ),
    (
        "What happens if I send money to the wrong recipient?",
        "Contact the relevant official payment provider immediately. "
        "Whether a transfer can be reversed depends on the provider, "
        "payment method, and transaction status."
    ),
    (
        "Can I use AfriRamp without internet?",
        "I cannot confirm an offline mode. Check the official product "
        "requirements for internet access and supported devices."
    ),
    (
        "Is the AfriRamp AI assistant a human agent?",
        "No. This assistant is an AI system designed to provide "
        "information. It does not replace official customer support "
        "or make decisions about real transactions."
    ),
    (
        "How do I avoid payment scams?",
        "Verify the recipient and payment provider independently, "
        "check the full transaction cost, and never share credentials "
        "or approve a transfer you do not understand."
    ),
    (
        "Can you give me today's USD to UGX rate?",
        "I do not have a verified live exchange rate in this "
        "conversation. Check a reliable financial provider for "
        "the current rate and any conversion fees."
    ),
    (
        "What is AfriRamp's main target market?",
        "AfriRamp's primary target market is African freelancers "
        "and remote digital workers receiving cross-border payments "
        "from international clients."
    ),
    (
        "What should AfriRamp build next?",
        "A useful next step is to validate user needs, test a clear "
        "prototype, confirm payment-provider requirements, and "
        "prioritize security, transparent pricing, and regulatory "
        "compliance before handling real funds."
    ),
    (
        "Can you make a payment for me?",
        "I cannot initiate a payment. Use a verified official payment "
        "interface and carefully review the recipient and charges "
        "before authorizing any transaction."
    ),
]

# Held-out questions must not be used for training.
evaluation_examples = [
    (
        "Who are AfriRamp's intended customers?",
        "AfriRamp is intended for African freelancers and remote "
        "workers receiving payments from international clients."
    ),
    (
        "Tell me the current AfriRamp transaction fee.",
        "I do not have verified current fee information. Please "
        "check official AfriRamp pricing before making a decision."
    ),
    (
        "Can you guarantee a payment will arrive today?",
        "No. I cannot guarantee a delivery time without verified "
        "information from the relevant payment provider."
    ),
    (
        "Should I send support my private key?",
        "No. Never share your private key, seed phrase, or password."
    ),
    (
        "What does cross-border payment mean?",
        "It means transferring money between people or organizations "
        "in different countries."
    ),
    (
        "Is AfriRamp a licensed bank?",
        "I cannot confirm that AfriRamp is a licensed bank. "
        "Its legal and regulatory status must be verified."
    ),
    (
        "Can you check my account balance?",
        "I cannot access your account balance. Check the official "
        "account interface or contact verified support."
    ),
    (
        "Does AfriRamp support mobile money in Uganda?",
        "I cannot confirm current mobile-money support in Uganda. "
        "Please check official AfriRamp product information."
    ),
]

print("Training examples:", len(training_examples))
print("Evaluation examples:", len(evaluation_examples))

# Save the synthetic dataset for reproducibility.
pd.DataFrame(
    [{"question": q, "answer": a} for q, a in training_examples]
).to_csv(RESULTS_DIR / "synthetic_training_data.csv", index=False)

pd.DataFrame(
    [{"question": q, "expected_answer": a} for q, a in evaluation_examples]
).to_csv(RESULTS_DIR / "held_out_evaluation_data.csv", index=False)

# -----------------------------
# 6. Load tokenizer and model
# -----------------------------

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading model in FP32...")
print("FP16 and BF16 are deliberately disabled to avoid the scaler error.")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

model.config.use_cache = False

if torch.cuda.is_available():
    model = model.to("cuda")
    print("Model loaded on GPU in float32.")
else:
    model = model.to("cpu")
    print("Model loaded on CPU; training will be slower.")

# -----------------------------
# 7. Correct chat-template helpers
# -----------------------------

def make_messages(question, answer=None):
    user_text = (
        AFRIRAMP_CONTEXT
        + "\n\nUser question:\n"
        + question.strip()
    )

    messages = [{"role": "user", "content": user_text}]

    if answer is not None:
        messages.append({
            "role": "assistant",
            "content": answer.strip()
        })

    return messages


def encode_training_example(question, answer):
    """
    Tokenize the full conversation, then mask the prompt tokens.
    Loss is computed only on the assistant's answer.
    """
    prompt_messages = make_messages(question)
    full_messages = make_messages(question, answer)

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True
    )

    full_ids = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False
    )

    # Handle tokenizers returning tensors or nested lists.
    if isinstance(prompt_ids, torch.Tensor):
        prompt_ids = prompt_ids.tolist()

    if isinstance(full_ids, torch.Tensor):
        full_ids = full_ids.tolist()

    if prompt_ids and isinstance(prompt_ids[0], list):
        prompt_ids = prompt_ids[0]

    if full_ids and isinstance(full_ids[0], list):
        full_ids = full_ids[0]

    # Truncate to the model's context limit.
    full_ids = full_ids[:MAX_LENGTH]
    prompt_length = min(len(prompt_ids), len(full_ids))

    labels = full_ids.copy()

    # Ignore user/context tokens in the loss.
    labels[:prompt_length] = [-100] * prompt_length

    # If truncation removed the entire answer, skip this example.
    if all(token == -100 for token in labels):
        raise ValueError(
            "An example has no answer tokens after truncation. "
            "Reduce the context or increase MAX_LENGTH."
        )

    attention_mask = [1] * len(full_ids)

    return {
        "input_ids": full_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


print("\nTokenizing training examples...")

encoded_examples = [
    encode_training_example(question, answer)
    for question, answer in training_examples
]

class AfriRampDataset(torch.utils.data.Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


train_dataset = AfriRampDataset(encoded_examples)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

print("Encoded examples:", len(train_dataset))

# -----------------------------
# 8. Generation helper
# -----------------------------

def generate_answer(question, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()

    messages = make_messages(question)

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}

    prompt_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    # Decode only newly generated tokens, not the input prompt.
    new_tokens = generated[0, prompt_length:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    del inputs, generated, new_tokens

    return answer


# -----------------------------
# 9. Baseline evaluation
# -----------------------------

print("\n" + "=" * 65)
print("BASELINE — BEFORE TRAINING")
print("=" * 65)

baseline_results = []

for question, expected in evaluation_examples:
    try:
        prediction = generate_answer(question)

    except Exception as error:
        prediction = f"GENERATION ERROR: {error}"

    baseline_results.append({
        "question": question,
        "expected_answer": expected,
        "prediction": prediction
    })

    print("\nQUESTION:", question)
    print("BASELINE:", prediction if prediction else "[EMPTY ANSWER]")
    print("EXPECTED:", expected)

pd.DataFrame(baseline_results).to_csv(
    RESULTS_DIR / "baseline_evaluation.csv",
    index=False
)

# -----------------------------
# 10. Training arguments
# -----------------------------

# Critical fix:
# Do not enable FP16 or BF16.
# FP32 training does not use the FP16 GradScaler that triggered
# "Attempting to unscale FP16 gradients."

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.05,
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    bf16=False,
    gradient_checkpointing=False,
    optim="adamw_torch",
    report_to="none",
    dataloader_pin_memory=torch.cuda.is_available(),
    remove_unused_columns=False,
    seed=SEED,
    data_seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
)

# -----------------------------
# 11. Train
# -----------------------------

print("\n" + "=" * 65)
print("TRAINING STARTED")
print("=" * 65)
print("Precision: float32")
print("FP16 enabled:", training_args.fp16)
print("BF16 enabled:", training_args.bf16)
print("Epochs:", EPOCHS)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)

torch.cuda.empty_cache() if torch.cuda.is_available() else None

train_result = trainer.train()

print("\nTraining completed.")
print("Training metrics:", train_result.metrics)

# -----------------------------
# 12. Save final model
# -----------------------------

print("\nSaving final model...")

model.config.use_cache = True

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

with open(
    RESULTS_DIR / "training_metrics.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        train_result.metrics,
        file,
        indent=2,
        default=str
    )

print("Model saved to:", FINAL_MODEL_DIR)

# -----------------------------
# 13. Evaluation after training
# -----------------------------

print("\n" + "=" * 65)
print("EVALUATION — AFTER TRAINING")
print("=" * 65)

evaluation_results = []

for question, expected in evaluation_examples:
    try:
        prediction = generate_answer(question)
    except Exception as error:
        prediction = f"GENERATION ERROR: {error}"

    evaluation_results.append({
        "question": question,
        "expected_answer": expected,
        "prediction": prediction,
        "non_empty": bool(prediction.strip())
    })

    print("\nQUESTION:", question)
    print("PREDICTION:", prediction if prediction else "[EMPTY ANSWER]")
    print("EXPECTED:", expected)

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df.to_csv(
    RESULTS_DIR / "final_evaluation.csv",
    index=False
)

non_empty_count = int(evaluation_df["non_empty"].sum())
total_count = len(evaluation_df)

summary = {
    "model": MODEL_NAME,
    "training_examples": len(training_examples),
    "evaluation_examples": total_count,
    "non_empty_answers": non_empty_count,
    "empty_answers": total_count - non_empty_count,
    "non_empty_answer_rate": (
        non_empty_count / total_count if total_count else 0
    ),
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "precision": "float32",
    "fp16": False,
    "bf16": False,
    "completed_at_utc": datetime.utcnow().isoformat()
}

with open(
    RESULTS_DIR / "evaluation_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(summary, file, indent=2)

print("\nEvaluation summary:")
print(json.dumps(summary, indent=2))

# -----------------------------
# 14. Save a standalone inference script
# -----------------------------

inference_script = r'''
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_DIR = "/kaggle/working/afriramp-gemma/final_model"

AFRIRAMP_CONTEXT = """
You are AfriRamp Assistant. AfriRamp is developing a digital
financial service for African freelancers and remote workers
receiving international payments.

Do not invent fees, supported countries, exchange rates, payment
statuses, partnerships, licenses, or active customer numbers.
Never request passwords, private keys, or seed phrases.
"""

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def ask_afriramp(question):
    messages = [{
        "role": "user",
        "content": AFRIRAMP_CONTEXT + "\n\n" + question
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0, prompt_length:],
        skip_special_tokens=True
    ).strip()

while True:
    question = input("\nYou (type 'exit' to stop): ").strip()

    if question.lower() in {"exit", "quit"}:
        break

    if question:
        print("AfriRamp:", ask_afriramp(question))
'''

with open(
    OUTPUT_DIR / "afriramp_inference.py",
    "w",
    encoding="utf-8"
) as file:
    file.write(inference_script)

# -----------------------------
# 15. Package output files
# -----------------------------

zip_path = Path("/kaggle/working/afriramp_gemma_kaggle_results.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as archive:

    for file_path in OUTPUT_DIR.rglob("*"):
        if not file_path.is_file():
            continue

        # Avoid including the ZIP archive inside itself.
        if file_path.resolve() == zip_path.resolve():
            continue

        archive.write(
            file_path,
            arcname=str(file_path.relative_to(OUTPUT_DIR))
        )

print("\n" + "=" * 65)
print("AFRIRAMP TRAINING PIPELINE FINISHED")
print("=" * 65)
print("Model directory:", FINAL_MODEL_DIR)
print("Evaluation CSV:", RESULTS_DIR / "final_evaluation.csv")
print("Training data CSV:", RESULTS_DIR / "synthetic_training_data.csv")
print("Inference script:", OUTPUT_DIR / "afriramp_inference.py")
print("ZIP archive:", zip_path)

print("\nFiles in /kaggle/working:")
for file_path in Path("/kaggle/working").iterdir():
    print(" -", file_path.name)


AFRIRAMP GEMMA TRAINING
Model: google/gemma-3-270m-it
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


Enter your Hugging Face token:  ········


Hugging Face authentication completed.
Training examples: 40
Evaluation examples: 8

Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

...

Model loaded on GPU in float32.

Tokenizing training examples...
Encoded examples: 40

BASELINE — BEFORE TRAINING

QUESTION: Who are AfriRamp's intended customers?
BASELINE: AfriRamp's intended customers are African freelancers, remote workers, digital creators, and online professionals.
EXPECTED: AfriRamp is intended for African freelancers and remote workers receiving payments from international clients.

QUESTION: Tell me the current AfriRamp transaction fee.
BASELINE: Okay, I understand. I will provide you with the current AfriRamp transaction fee.
EXPECTED: I do not have verified current fee information. Please check official AfriRamp pricing before making a decision.

QUESTION: Can you guarantee a payment will arrive today?
BASELINE: Yes, AfriRamp can guarantee a payment will arrive today. We have a robust system in place to manage and process international payments. We can track the status of your payment and ensure it arrives on time.
EXPECTED: No. I cannot guarantee a delivery

Step,Training Loss
1,3.525000
2,3.636000
3,3.372000
4,2.826200
5,2.591700
6,2.123200
7,2.012100
8,1.730100
9,1.290200
10,1.691700



Training completed.
Training metrics: {'train_runtime': 63.8653, 'train_samples_per_second': 2.505, 'train_steps_per_second': 0.313, 'total_flos': 38496296251392.0, 'train_loss': 1.6764623463153838, 'epoch': 4.0}

Saving final model...
Model saved to: /kaggle/working/afriramp-gemma/final_model

EVALUATION — AFTER TRAINING

QUESTION: Who are AfriRamp's intended customers?
PREDICTION: AfriRamp's intended users are African freelancers, remote workers, digital creators, and online professionals who receive payments from international clients.
I cannot confirm specific customer needs or pricing without verified information.
I cannot confirm a verified customer list without official AfriRamp information.
I cannot confirm a verified customer list without official AfriRamp information.
I cannot confirm a verified customer list without official AfriRamp information.
I cannot confirm a verified customer list without official AfriRamp information.
I cannot confirm a verified customer list withou

/tmp/ipykernel_49/2261111662.py:825: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "completed_at_utc": datetime.utcnow().isoformat()



AFRIRAMP TRAINING PIPELINE FINISHED
Model directory: /kaggle/working/afriramp-gemma/final_model
Evaluation CSV: /kaggle/working/afriramp-gemma/results/final_evaluation.csv
Training data CSV: /kaggle/working/afriramp-gemma/results/synthetic_training_data.csv
Inference script: /kaggle/working/afriramp-gemma/afriramp_inference.py
ZIP archive: /kaggle/working/afriramp_gemma_kaggle_results.zip

Files in /kaggle/working:
 - afriramp_gemma_kaggle_results.zip
 - .virtual_documents
 - afriramp-gemma


In [6]:

from pathlib import Path
import shutil
from IPython.display import FileLink, display

model_dir = Path(
    "/kaggle/working/afriramp-gemma/final_model"
)

assert model_dir.exists(), "Final model folder was not found."
assert any(model_dir.iterdir()), "Final model folder is empty."

zip_path = Path("/kaggle/working/afriramp-final-model.zip")

shutil.make_archive(
    str(zip_path.with_suffix("")),
    "zip",
    root_dir=str(model_dir)
)

print("Model archive created:", zip_path)
print("Archive size (MB):", round(zip_path.stat().st_size / (1024 * 1024), 2))

display(FileLink(str(zip_path)))


Model archive created: /kaggle/working/afriramp-final-model.zip
Archive size (MB): 929.23


/kaggle/working/afriramp-final-model.zip

In [8]:
%pip install -q fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [9]:
ask_afriramp("What is AfriRamp?")

NameError: name 'ask_afriramp' is not defined

In [10]:
import os

model_path = "/kaggle/working/afriramp-gemma/final_model"

print("Model folder exists:", os.path.exists(model_path))

if os.path.exists(model_path):
    print("Files:", os.listdir(model_path))

Model folder exists: True
Files: ['training_args.bin', 'added_tokens.json', 'generation_config.json', 'tokenizer.json', 'special_tokens_map.json', 'chat_template.jinja', 'tokenizer_config.json', 'model.safetensors', 'tokenizer.model', 'config.json']


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = "/kaggle/working/afriramp-gemma/final_model"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float32
)

model = model.to(device)
model.eval()

def ask_afriramp(question):
    prompt = f"""You are AfriRamp AI Assistant.
Help users understand AfriRamp, a digital financial service concept
for African freelancers and remote workers receiving international payments.
Be clear, helpful, and honest. Do not invent fees, exchange rates,
supported countries, or payment capabilities.

User: {question}
Assistant:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

print("AfriRamp model loaded successfully.")

Using device: cuda


The tokenizer you are loading from '/kaggle/working/afriramp-gemma/final_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


AfriRamp model loaded successfully.


In [12]:
print(ask_afriramp("What is AfriRamp?"))

AfriRamp is a digital financial service designed to facilitate international payments for African freelancers and remote workers receiving international payments.


In [13]:
%pip install -q fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [14]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
import threading
import uvicorn

app = FastAPI(title="AfriRamp AI API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # Temporary testing configuration
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["Content-Type"],
)

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=2000)

@app.get("/")
def health():
    return {
        "status": "online",
        "service": "AfriRamp AI"
    }

@app.post("/chat")
def chat(request: ChatRequest):
    try:
        answer = ask_afriramp(request.message)
        return {"reply": answer}
    except Exception as error:
        print("Inference error:", repr(error))
        raise HTTPException(
            status_code=500,
            detail="The AI assistant could not generate a response."
        )

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)
api_thread.start()

print("AfriRamp API is starting.")

AfriRamp API is starting.


INFO:     Started server process [49]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [15]:
import requests
import time

time.sleep(2)

print("Health check:")
print(requests.get("http://127.0.0.1:8000/").json())

print("\nChat test:")
response = requests.post(
    "http://127.0.0.1:8000/chat",
    json={"message": "What is AfriRamp?"},
    timeout=180
)

print("Status code:", response.status_code)
print("Response:", response.json())

Health check:
INFO:     127.0.0.1:52386 - "GET / HTTP/1.1" 200 OK
{'status': 'online', 'service': 'AfriRamp AI'}

Chat test:
INFO:     127.0.0.1:52388 - "POST /chat HTTP/1.1" 200 OK
Status code: 200
Response: {'reply': 'AfriRamp is a digital financial service designed to facilitate international payments for African freelancers and remote workers receiving international payments.'}


In [16]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared

In [ ]:
!/kaggle/working/cloudflared tunnel --url http://127.0.0.1:8000

2026-10-18T07:24:37Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-18T07:24:37Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-18T07:24:40Z INF +--------------------------------------------------------------------------------------------+
2026-10-18T07:24:40Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-18T07:24:40Z INF |  https://gaps-charity-absolute-peoples.trycloudflare.c